# Run Coding Assistant — Air-Gapped (Closed Network)

This notebook demonstrates the coding assistant experience in a **fully air-gapped environment** with only **3 local MCP servers** active. No internet required.

**Environment**: Closed network (no internet access)

| Tool | Server | Use Case | Replaces |
|------|--------|----------|----------|
| Code Sandbox | Local | Execute and verify code | *(same)* |
| Codebase Search | Local AI | Semantic code search | Context7 (library docs) |
| Repo Docs | Local AI | Internal documentation Q&A | SearXNG (web search) |

> **Compare with**: `2_run_public_coding_assistant.ipynb` — same scenario with all 5 tools (internet required).

**Scenario**: Same task — Add a "daily specials" feature to the `cafe-order-system`.

---

### Public vs Air-Gapped: Tool Mapping

| Task | Public (Internet) | Air-Gapped (This notebook) |
|------|-------------------|----------------------------|
| Library docs lookup | Context7 (official docs) | **Codebase Search** (find patterns in existing code) |
| Web search for best practices | SearXNG (blogs, SO) | **Repo Docs** (internal architecture/onboarding docs) |
| Code execution | Code Sandbox | **Code Sandbox** (identical) |
| Internal code search | Codebase Search | **Codebase Search** (identical) |
| Internal docs Q&A | Repo Docs | **Repo Docs** (identical) |

## 1. Verify MCP Server Connectivity (Air-Gapped Only)

In [ ]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

ALL_ROUTES = {}
for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        ALL_ROUTES[name] = f"https://{host}/mcp"

# Air-gapped: only local servers
AIRGAP_SERVERS = ["mcp-code-sandbox", "mcp-codebase-search", "mcp-repo-docs"]
INTERNET_SERVERS = ["mcp-context7", "mcp-searxng"]

ROUTES = {k: v for k, v in ALL_ROUTES.items() if k in AIRGAP_SERVERS}

print("MCP Servers (Air-Gapped Mode — 3 local servers only):")
print("=" * 65)
for name in AIRGAP_SERVERS:
    url = ROUTES.get(name, "NOT DEPLOYED")
    status = "READY" if name in ROUTES else "MISSING"
    print(f"  [{status}] {name:<25} {url}")

print("")
print("  DISABLED (require internet):")
for name in INTERNET_SERVERS:
    print(f"  [SKIP]  {name:<25} (blocked in air-gapped)")

missing = [n for n in AIRGAP_SERVERS if n not in ROUTES]
if missing:
    print(f"\n  WARNING: Missing: {missing}. Run deploy notebook first.")
else:
    print(f"\n  All 3 air-gapped servers ready.")

## 2. Generate IDE Configuration (Air-Gapped Only)

Generate MCP config with **only the 3 local servers**. Context7 and SearXNG are excluded.

In [ ]:
cursor_config = {"mcpServers": {}}
opencode_config = {"$schema": "https://opencode.ai/config.json", "mcp": {}}

for name, url in ROUTES.items():
    short = name.replace("mcp-", "")
    cursor_config["mcpServers"][short] = {"url": url}
    opencode_config["mcp"][short] = {"type": "remote", "url": url}

print("=== .cursor/mcp.json (AIR-GAPPED — 3 local servers only) ===")
print(json.dumps(cursor_config, indent=2))

print("")
print("=== opencode.json (AIR-GAPPED — 3 local servers only) ===")
print(json.dumps(opencode_config, indent=2))

print("")
print("NOTE: Context7 and SearXNG are NOT included.")
print("The agent will rely on Codebase Search and Repo Docs instead.")

## 3. AGENTS.md — Agent Harness (same as Public)

`AGENTS.md` works **identically in air-gapped mode**. The agent reads the local file from the project root and follows the same plan-execute-verify inner loop — no internet required.

See `2_run_public_coding_assistant.ipynb` Section 3 for the full walkthrough and AGENTS.md content.

## 4. Same Scenario: Add "Daily Specials" (Air-Gapped)

Same task as the public notebook, but using **only local tools**.

---

### Step 1: Understand the existing codebase

**Tool: Codebase Search** — "How are menu items structured?" (identical to public version)

In [ ]:
init = json.dumps({"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}})

url = ROUTES.get("mcp-codebase-search", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_code","arguments":{"query":"menu item model class definition","top_k":2}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Codebase Search > search_code")
    print("Query: 'menu item model class definition'")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:800])
            except: pass
else:
    print("mcp-codebase-search not available")

### Step 2: Find FastAPI route patterns (replaces Context7)

**Tool: Codebase Search** — "How are existing API routes defined?"

Instead of looking up official FastAPI docs via Context7, the agent searches **existing code patterns** in the project.

In [ ]:
url = ROUTES.get("mcp-codebase-search", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_code","arguments":{"query":"FastAPI router GET endpoint with query parameters","top_k":3}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Codebase Search > search_code (REPLACES Context7)")
    print("Query: 'FastAPI router GET endpoint with query parameters'")
    print("Strategy: Learn from existing code patterns instead of official docs")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:800])
            except: pass
else:
    print("mcp-codebase-search not available")

### Step 3: Search internal docs for conventions (replaces SearXNG)

**Tool: Repo Docs** — "What are the project coding conventions and patterns?"

Instead of searching the web via SearXNG, the agent consults **internal documentation** — onboarding guide, architecture docs, etc.

In [ ]:
url = ROUTES.get("mcp-repo-docs", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_docs","arguments":{"query":"coding conventions and project structure for new features","top_k":3}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Repo Docs > search_docs (REPLACES SearXNG)")
    print("Query: 'coding conventions and project structure for new features'")
    print("Strategy: Internal docs replace web search for best practices")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:600])
            except: pass
else:
    print("mcp-repo-docs not available")

### Step 4: Execute and verify the implementation

**Tool: Code Sandbox** — Run the generated code (identical to public version).

In [ ]:
url = ROUTES.get("mcp-code-sandbox", "")
if url:
    test_code = '''import json\nspecials = [{"name": "아메리카노", "price": 3500, "original_price": 4500, "discount": "22%"},\n            {"name": "크루아상", "price": 3000, "original_price": 4000, "discount": "25%"}]\nprint(json.dumps({"date": "2026-06-16", "specials": specials}, ensure_ascii=False, indent=2))'''
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"execute_code","arguments":{"code":test_code,"language":"python"}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Code Sandbox > execute_code")
    print("Action: Run generated daily specials API response")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", ""))
            except: pass
else:
    print("mcp-code-sandbox not available")

## 5. Comparison: Public vs Air-Gapped

| Step | Task | Public Mode | Air-Gapped Mode | Quality Impact |
|------|------|-------------|-----------------|----------------|
| 1 | Understand codebase | Codebase Search | Codebase Search | **Identical** |
| 2 | Library docs | Context7 (official) | Codebase Search (patterns) | Slightly less comprehensive |
| 3 | Best practices | SearXNG (web) | Repo Docs (internal) | Internal-focused, no community |
| 4 | Internal conventions | Repo Docs | Repo Docs | **Identical** |
| 5 | Code execution | Code Sandbox | Code Sandbox | **Identical** |

### Key Takeaways

**What works identically in air-gapped:**
- Code search, code execution, internal documentation — no degradation

**What changes in air-gapped:**
- Library documentation: relies on existing code patterns instead of official docs
- Web search: replaced by internal documentation (architecture, onboarding guides)
- Agent quality remains high for **project-specific tasks** (which is the primary use case)

**Recommendation:**
- For air-gapped environments, index additional library docs (e.g., FastAPI, SQLAlchemy markdown) into the Repo Docs ConfigMap to close the gap with Context7.

## Next Steps

- `../2_maas/` — Add MaaS gateway for centralized auth and API key management
- `../4_control/` — Configure rate limiting and subscription policies